In [9]:
import os
from pathlib import Path

import polars as pl

# ===== MODE =====
USE_FULL_DATA = False          # default: False (pakai preview biar ringan)
PREVIEW_N_ROWS = 100_000       # ukuran preview (sesuaikan)
RANDOM_SEED = 42               # buat sampling konsisten

# ===== PATHS (sesuaikan dengan struktur repo kamu) =====
# Disarankan preview disimpan di folder data/preview (di root repo), lalu di-commit ke GitHub
PREVIEW_PATH = Path("data/preview/dgph_preview_100k.parquet")

# Full dataset disimpan lokal saja (JANGAN di-commit); ini contoh sesuai error path kamu
FULL_DATA_DIR = Path("notebooks/data/raw/huggingface/akhyar016__dgph-covid19-social-media")

# Folder parquet yang akan discan (HF auto-converted biasanya ada folder parquet)
# Nanti kita cari otomatis di bawah
print("USE_FULL_DATA =", USE_FULL_DATA)
print("PREVIEW_PATH  =", PREVIEW_PATH.resolve())
print("FULL_DATA_DIR =", FULL_DATA_DIR.resolve())


USE_FULL_DATA = False
PREVIEW_PATH  = /home/jovyan/work/notebooks/data/preview/dgph_preview_100k.parquet
FULL_DATA_DIR = /home/jovyan/work/notebooks/notebooks/data/raw/huggingface/akhyar016__dgph-covid19-social-media


In [ ]:
try:
    from huggingface_hub import snapshot_download
except ModuleNotFoundError:
    # Kalau kernel kamu benar, ini harusnya bekerja.
    # Kalau masih error, problemnya bukan kode, tapi environment/kernel.
    import sys
    !{sys.executable} -m pip install -U huggingface_hub

    from huggingface_hub import snapshot_download

DATASET_ID = "akhyar016/dgph-covid19-social-media"

FULL_DATA_DIR.mkdir(parents=True, exist_ok=True)

snapshot_path = snapshot_download(
    repo_id=DATASET_ID,
    repo_type="dataset",
    local_dir=str(FULL_DATA_DIR),
    local_dir_use_symlinks=False,
    resume_download=True,
    max_workers=1,
)

print("Snapshot downloaded to:", snapshot_path)
print("Listing top-level files:")
print(sorted(os.listdir(snapshot_path))[:50])


In [ ]:
def find_parquet_files(root: Path):
    if not root.exists():
        return []
    return sorted([p for p in root.rglob("*.parquet") if p.is_file()])

# Pastikan folder preview ada
PREVIEW_PATH.parent.mkdir(parents=True, exist_ok=True)

parquet_files = find_parquet_files(FULL_DATA_DIR)
print("Found parquet files:", len(parquet_files))

if len(parquet_files) == 0:
    raise FileNotFoundError(
        f"Tidak menemukan .parquet di {FULL_DATA_DIR}. "
        "Kalau kamu belum download full dataset, jalankan cell download dulu "
        "atau pastikan path FULL_DATA_DIR benar."
    )

# Polars lazy scan semua parquet (schema konsisten)
lf_full = pl.scan_parquet([str(p) for p in parquet_files])

# Bikin preview (random sample)
# Catatan: sampling di LazyFrame butuh collect di akhir.
df_preview = (
    lf_full
    .sample(n=PREVIEW_N_ROWS, with_replacement=False, seed=RANDOM_SEED)
    .collect()
)

print(df_preview.shape)
print(df_preview.head(3))

df_preview.write_parquet(PREVIEW_PATH)
print("Preview saved to:", PREVIEW_PATH)
print("Preview size (MB):", round(PREVIEW_PATH.stat().st_size / (1024*1024), 2))


In [ ]:
def get_lazyframe():
    # Prioritas: kalau USE_FULL_DATA True dan file full ada -> full
    if USE_FULL_DATA:
        parquet_files = find_parquet_files(FULL_DATA_DIR)
        if len(parquet_files) == 0:
            raise FileNotFoundError(
                f"USE_FULL_DATA=True tapi tidak ada parquet di {FULL_DATA_DIR}."
            )
        print(f"Loading FULL dataset from {FULL_DATA_DIR} (parquet files: {len(parquet_files)})")
        return pl.scan_parquet([str(p) for p in parquet_files])

    # Default: preview
    if not PREVIEW_PATH.exists():
        raise FileNotFoundError(
            f"Preview tidak ditemukan di {PREVIEW_PATH}. "
            "Jalankan cell pembuatan preview dulu, atau set USE_FULL_DATA=True."
        )
    print(f"Loading PREVIEW dataset from {PREVIEW_PATH}")
    return pl.scan_parquet(str(PREVIEW_PATH))

lf = get_lazyframe()
lf
